# Data Inventory and Quality Audit

## Goal
Establish file-level provenance, schema, temporal coverage, missingness, duplication, and obvious naming inconsistencies before substantive analysis. This notebook intentionally makes no causal claims.

## Context & Methods

### Key assumptions
- Files in `database/` are treated as source exports and are not modified.
- A paired `.txt` file, when present, is treated as the SQL/query provenance record.
- Filename claims such as `since-2008` are checked against observed dates rather than trusted.

In [ ]:
from pathlib import Path
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

def locate_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / "database").exists() and (candidate / "notebooks").exists():
            return candidate
        nested = candidate / "stack_exchange_analysis"
        if (nested / "database").exists() and (nested / "notebooks").exists():
            return nested
    raise FileNotFoundError("Could not locate stack_exchange_analysis project root.")

PROJECT_ROOT = locate_project_root()
DATA_DIR = PROJECT_ROOT / "database"
ANALYSIS_DIR = PROJECT_ROOT / "analysis"
ANALYSIS_DIR.mkdir(exist_ok=True)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from analysis_utils import read_csv_flexible, parse_best_date_column, drop_incomplete_last_period, save_figure

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
print(f"Project root: {PROJECT_ROOT}")

### 1. Inventory source files

In [ ]:
files = sorted([p for p in DATA_DIR.iterdir() if p.is_file()])
inventory = pd.DataFrame({
    "file": [p.name for p in files],
    "suffix": [p.suffix.lower() for p in files],
    "size_mb": [p.stat().st_size / 1_000_000 for p in files],
})
inventory

### 2. Profile CSV files

In [ ]:
records = []
for path in sorted(DATA_DIR.glob("*.csv")):
    try:
        df = read_csv_flexible(path)
        date_col, parsed = parse_best_date_column(df)
        record = {
            "file": path.name,
            "rows": len(df),
            "columns": df.shape[1],
            "duplicate_rows": int(df.duplicated().sum()),
            "missing_cells_pct": float(df.isna().mean().mean() * 100),
            "date_column": date_col,
            "date_min": parsed.min() if parsed is not None else pd.NaT,
            "date_max": parsed.max() if parsed is not None else pd.NaT,
            "paired_query": (path.with_suffix('.txt')).exists(),
        }
        records.append(record)
    except Exception as exc:
        records.append({"file": path.name, "error": repr(exc)})

profile = pd.DataFrame(records)
profile

### 3. Flag temporal naming inconsistencies

In [ ]:
checks = profile.copy()
checks["claims_since_2008"] = checks["file"].str.contains("since-2008", case=False, na=False)
checks["observed_start_year"] = pd.to_datetime(checks["date_min"], errors="coerce").dt.year
checks["start_year_mismatch"] = checks["claims_since_2008"] & checks["observed_start_year"].notna() & (checks["observed_start_year"] > 2008)
checks.loc[checks["claims_since_2008"], ["file", "observed_start_year", "start_year_mismatch"]]

### 4. Inspect query provenance

In [ ]:
query_records = []
for path in sorted(DATA_DIR.glob("*.txt")):
    text = path.read_text(encoding="utf-8", errors="replace")
    lower = text.lower()
    query_records.append({
        "query_file": path.name,
        "chars": len(text),
        "mentions_stackoverflow": "stackoverflow" in lower,
        "mentions_pt_endpoint": "data.stackexchange.com/pt/" in lower,
        "mentions_postswithdeleted": "postswithdeleted" in lower,
        "mentions_posts": " posts" in lower or "..posts" in lower,
    })
query_audit = pd.DataFrame(query_records)
query_audit

### 5. Export audit tables

In [ ]:
inventory.to_csv(ANALYSIS_DIR / "data_file_inventory.csv", index=False)
profile.to_csv(ANALYSIS_DIR / "data_quality_profile.csv", index=False)
query_audit.to_csv(ANALYSIS_DIR / "query_provenance_audit.csv", index=False)
print("Audit tables written to analysis/.")

## Takeaways
Use the generated audit tables to resolve source/site ambiguities before combining datasets. In particular, do not infer site identity from filenames alone.